# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library, referencing all data entities by their `@id` values, as per Croissant best practices.

### Dataset Source
The Croissant schema for this dataset is available at:
`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`. We will use the Croissant schema URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

We will list out all available record sets, then for each, show its fields (with their `@id`s), and note key attribute columns available in the data.

In [ ]:
# Explore the record sets defined in the dataset metadata
rs_entities = metadata.record_sets
if not rs_entities:
    print('No record sets defined in Croissant schema. Attempting to infer available data entries by examining Data Distribution...')
    # Fallback if no record sets present: Show content of distributions
    for dist in metadata.distribution:
        print(f"Distribution: @id: {dist['@id']} (hint: usually a file/data resource)")
else:
    for rs in rs_entities:
        print(f"RecordSet: @id: {rs['@id']}")
        if 'field' in rs:
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            for field in fields:
                print(f"  Field: @id: {field.get('@id', str(field))} (name: {field.get('name', '')})")
        if 'column' in rs:
            columns = rs['column'] if isinstance(rs['column'], list) else [rs['column']]
            for col in columns:
                print(f"  Column: @id: {col.get('@id', str(col))}")
if not rs_entities:
    print("Assuming one data table per 'distribution', to be referenced by its @id below.")

## 3. Data Extraction
Load data from a specific record set (or distribution) into a pandas DataFrame for analysis.

We will use the `@id` of the data resource as required by the schema, referencing them by variable.

In [ ]:
# List all available distributions and load them by their @id
# This FAIR² dataset contains two primary distributions (data resources).
distribution_ids = [
    'http://nexus-delta.data-vitae-prd.svc.cluster.local/v1/resources/frontiers/7853015/_/8336ac61-9308-403f-8df3-28e120cc98f3',
    'http://nexus-delta.data-vitae-prd.svc.cluster.local/v1/resources/frontiers/7853015/_/8e507442-660d-4cfe-b2d9-f805d7abe725'
]

# For demonstration, let's use the first distribution @id as our example primary data table
main_distribution_id = distribution_ids[0]
dataframes = {}

for dist_id in distribution_ids:
    print(f"Loading data from distribution @id: {dist_id}")
    try:
        records = list(dataset.records(record_set=dist_id))
        df = pd.DataFrame(records)
        dataframes[dist_id] = df
        print(f"Loaded shape: {df.shape}")
    except Exception as e:
        print(f"  Could not load records for {dist_id}: {e}")

# Preview columns of the main data resource
if main_distribution_id in dataframes:
    print('Columns in main distribution:')
    print(dataframes[main_distribution_id].columns.tolist())
    display(dataframes[main_distribution_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
- Filtering records based on numeric or categorical fields
- Handling missing or outlier values
- Normalizing numeric fields
- Grouping/categorizing data as appropriate

For demonstration, we pick a numeric field and a group field by their column `@id`s.

In [ ]:
# Choose a numeric field and a group field from the dataset for demonstration
# For example, let's use 'log_likelihood' (from regression results) if present, and 'ward' (a location/grouping field)

numeric_field = None
group_field = None
df = dataframes.get(main_distribution_id, pd.DataFrame())

# Guess candidate columns
for c in df.columns:
    if 'log' in c.lower() and 'likelihood' in c.lower():
        numeric_field = c
    if 'ward' in c.lower():
        group_field = c

# If not found, pick 'coeff' or any numeric columns by name
if not numeric_field:
    for c in df.columns:
        if 'coeff' in c.lower() or 'estimate' in c.lower():
            numeric_field = c
            break
if not numeric_field:
    # fallback: pick any float/int column
    for c in df.select_dtypes(include=['float', 'int']).columns:
        numeric_field = c
        break

if not group_field:
    # Try other likely group columns
    for c in df.columns:
        if 'county' in c.lower() or 'region' in c.lower():
            group_field = c
            break

print(f"Using numeric field: {numeric_field}")
print(f"Using group field: {group_field}")

# Proceed with EDA if field present
if numeric_field in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field]):
    # Remove outliers: Keep numeric_field > 10 (arbitrary threshold for demonstration)
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold} (count: {filtered_df.shape[0]}):")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group_field if it exists
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print("Could not identify a suitable numeric field for EDA. Check data preview above.")

## 5. Visualization
Visualize distributions or relationships in the data using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple distribution plot for the selected numeric field
if numeric_field and numeric_field in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

# If group_field available, a boxplot
if group_field and group_field in df.columns and numeric_field in df.columns:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.ylabel(numeric_field)
    plt.xlabel(group_field)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

This notebook demonstrated loading a Croissant-annotated FAIR² dataset with `mlcroissant`, explored the schema and contents by referencing all data entities with their `@id` field, and performed basic EDA and visualization. For further analysis, consult the Croissant schema to identify additional record sets and @id fields for deeper or more tailored exploration and downstream machine learning workflows.